# 03 · Kaggle CNN training

Enable a GPU accelerator. Attach the saved output from notebook 01 and your private reviewed `duplicate_decisions.csv`. The notebook stops if review is incomplete or an official-boundary conflict exists.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/mamta-kumari28/Chest-X-Ray-Diagnosis-using-Deep-Learning-.git"
REPO_REF = "phase2/final-validation-fixes"  # Change to main after the PR is merged.
ROOT = Path("/kaggle/working/covid-xray-project")
OUTPUT_ROOT = Path("/kaggle/working/covid_xray_outputs")

if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Outputs:", OUTPUT_ROOT)

In [ ]:
def find_one(name, roots=(Path("/kaggle/working"), Path("/kaggle/input"))):
    matches = []
    for root in roots:
        matches.extend(root.glob(f"**/{name}"))
    matches = [path for path in matches if ".git" not in path.parts]
    if not matches:
        raise FileNotFoundError(f"Could not find {name}. Attach the previous notebook output as a Kaggle input.")
    print(name, "->", matches[0])
    return matches[0]

VALIDATED = find_one("validated_metadata.csv")
DECISIONS = find_one("duplicate_decisions.csv")

In [ ]:
import pandas as pd
from covid_xray.review import build_duplicate_review, enforce_official_duplicate_boundary

audited = pd.read_csv(VALIDATED, dtype=str, keep_default_na=False)
decisions = pd.read_csv(DECISIONS, dtype=str, keep_default_na=False)
review = build_duplicate_review(audited, decisions)
groups = review.drop_duplicates("group_id")
unresolved = groups[(groups["evidence_type"] == "perceptual") & (groups["review_status"] == "unresolved")]
if not unresolved.empty:
    raise ValueError(f"Human duplicate review is incomplete: {len(unresolved)} perceptual groups remain unresolved.")
enforce_official_duplicate_boundary(review)
print("Duplicate review complete and official boundary guard passed.")

In [ ]:
# Confirm this only after checking source-specific permissions documented in docs/DATASET.md.
SOURCE_PERMISSIONS_CONFIRMED = False
if not SOURCE_PERMISSIONS_CONFIRMED:
    raise RuntimeError("Set SOURCE_PERMISSIONS_CONFIRMED=True only after recording the source permission decision.")

In [ ]:
from covid_xray.data import build_manifest

valid = audited[audited["validation_status"].eq("valid")].copy()
index = valid.rename(columns={
    "resolved_path": "path",
    "original_partition": "original_split",
})[["path", "patient_id", "label", "sha256", "perceptual_hash", "original_split"]]

PROCESSED = OUTPUT_ROOT / "processed"
MODELS = OUTPUT_ROOT / "models"
PROCESSED.mkdir(parents=True, exist_ok=True)
MANIFEST = PROCESSED / "manifest.csv"
manifest = build_manifest(index, val_size=0.15, review=decisions)
manifest.to_csv(MANIFEST, index=False)
display(pd.crosstab(manifest["split"], manifest["label"], margins=True))

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")

subprocess.run([
    sys.executable, "-m", "covid_xray.train",
    "--manifest", str(MANIFEST), "--output", str(MODELS),
    "--epochs", "30", "--batch-size", "32",
], check=True)
print("Best model:", MODELS / "model.keras")

Use **Save Version → Save & Run All**, then attach this saved output to notebook 04. Keep the model private until its redistribution permission is confirmed.